## Introduction

### Need of Image Generation

In [ ]:
'''
| Application                            | How Walmart Can Use It                                                                     | Business Benefit                                             |
| -------------------------------------- | ------------------------------------------------------------------------------------------ | ------------------------------------------------------------ |
| **Product Image Generation**           | Create product images with different backgrounds, lighting, and settings                   | Faster catalog creation and reduced photography costs        |
| **Virtual Try-On**                     | Generate images of customers wearing apparel, glasses, or accessories locally in stores    | Better customer experience while protecting customer privacy |
| **Store Layout Visualization**         | Create shelf arrangements and store planograms                                             | Improve merchandising and inventory placement                |
| **Marketing Content**                  | Generate banners, social media creatives, and promotional posters                          | Faster campaign creation                                     |
| **Packaging Design**                   | Produce multiple packaging concepts for private-label products                             | Accelerate product development                               |
| **Furniture/Home Decor Visualization** | Show furniture in different room settings                                                  | Increase customer confidence before purchase                 |
| **Synthetic Training Data**            | Generate labeled images for computer vision models (shelf monitoring, inventory detection) | Improve AI model accuracy while reducing manual labeling     |
| **Damage Simulation**                  | Generate damaged packaging or products for quality inspection model training               | Better defect detection systems                              |
| **Localization**                       | Create culturally relevant promotional images for different regions                        | Personalized marketing at scale                              |
| **Inventory Visualization**            | Generate expected shelf images for comparison with real shelf photos                       | Detect stock-outs and misplaced items                        |
'''

### Popular offline image generation models

### Why offline models are attractive for Walmart

## Implementing offline models for Image Generation

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import torch
from diffusers import DiffusionPipeline
pipe = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
pipe.to("cuda")

In [ ]:
prompt = "A futuristic city skyline at sunset"

images = pipe(prompt=prompt).images

print(len(images))
# for img in images:
#     plt.imshow(img)
#     plt.show()

#images

In [ ]:
images[0]

In [ ]:
prompt = "A futuristic city skyline at sunset"

images = pipe(prompt=prompt).images[0]

images


In [ ]:
# Get the image dimensions (height, width, and optionally color channels)
import numpy as np
numpy_image = np.array(images)
height, width, *channels = numpy_image.shape

# Print the resolution
print(f"Image Resolution: {width}x{height} pixels")

# Prompt Engineering: Improving Results

In [ ]:
# Start with the subject
prompt = "A cat"
images = pipe(prompt=prompt).images[0]
images

In [ ]:
prompt = "A realistic painting of a cat sitting by the window"
images = pipe(prompt=prompt).images[0]
images

In [ ]:
prompt = "A realistic painting of a cat looking right and sitting by the window"
images = pipe(prompt=prompt).images[0]
images

In [ ]:
# Add style or details
prompt = "A realistic painting of a cat sitting by the window, watercolor"
images = pipe(prompt=prompt).images[0]
images

In [ ]:
#experimenting with styles ("digital art", "watercolor", "photorealistic", etc.)
prompt = "A realistic painting of a cat sitting by the window, digital art"
images = pipe(prompt=prompt).images[0]
images

In [ ]:
'''
TRY OUT THESE PROMPTS:

prompt = "portrait of a smiling girl"
prompt = "portrait of a sad girl"
prompt = "portrait of a angry girl"
prompt = "portrait of a girl laughing"
'''

# Parameters and Customization

## Seeds: Control randomness. 

In [ ]:
prompt = "A realistic painting of a cat sitting by the window"
images = pipe(prompt=prompt, seed=42).images[0]
images

In [ ]:
prompt = "A realistic painting of a cat sitting by the window"
images = pipe(prompt=prompt, seed=42).images[0]
images

In [ ]:
prompt = "A realistic painting of a cat sitting by the window"
images = pipe(prompt=prompt, seed=42).images[0]
images

In [ ]:
# A better prompt alongside seed can be a good way to make results consistent but not exact
prompt = "A realistic painting of a cat looking right and sitting by the left side of window"
images = pipe(prompt=prompt, seed=42).images[0]
images

In [ ]:
prompt = "A realistic painting of a cat looking right and sitting by the left side of window"
images = pipe(prompt=prompt, seed=42).images[0]
images

In [ ]:
prompt = "A realistic painting of a cat looking right and sitting by the left side of window"
images = pipe(prompt=prompt, seed=314).images[0]
images

## Steps: More steps, better detail (but slower)

In [ ]:
prompt = "A realistic portrait of a girl"
images = pipe(prompt=prompt, num_inference_steps=1).images[0]
images

In [ ]:
prompt = "A realistic portrait of a girl"
images = pipe(prompt=prompt, num_inference_steps=50).images[0]
images

In [ ]:
prompt = "A realistic portrait of a girl"
images = pipe(prompt=prompt, num_inference_steps=30).images[0]
images

### Resolution

In [ ]:
# Set image resolution (width, height) - must be divisible by 8

#width, height = 128, 128
#width, height = 512, 512
#width, height = 1024, 512
#width, height = 512, 1024
width, height = 1024, 1024

prompt = "Potrait of a person eating apple"
images = pipe(prompt=prompt, width=width, height=height).images[0]
images

### Logo Generation

In [ ]:
prompt = "A tech startup logo with a sleek circuit pattern forming the letter ‘T,’ sharp lines, and a metallic gradient finish, modern and minimalist, vector style, white background. The logo must have the text 'TECH'"
images = base(prompt=prompt).images[0]
images

## Interactive demo

In [ ]:
# Interactive demo with widgets (requires ipywidgets)
# !pip install -q ipywidgets
from ipywidgets import interact, Text, FloatSlider
@interact(prompt=Text(value='A realistic portrait'))
def generate_demo(prompt):
    img = pipe(prompt).images[0]
    display(img)

# Mixutre of Experts

In [ ]:
from diffusers import DiffusionPipeline
import torch

# load both base & refiner
base = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, variant="fp16", use_safetensors=True
)

base.to("cuda")
refiner = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    text_encoder_2=base.text_encoder_2,
    vae=base.vae,
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16",
)
refiner.to("cuda")

# Define how many steps and what % of steps to be run on each experts (80/20) here
n_steps = 40
high_noise_frac = 0.8

prompt = "A majestic lion jumping from a big stone at night"

# run both experts
image = base(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_end=high_noise_frac,
    output_type="latent",
).images

image = refiner(
    prompt=prompt,
    num_inference_steps=n_steps,
    denoising_start=high_noise_frac,
    image=image,
).images[0]

In [ ]:
image

# Limitations

In [ ]:
## In-processing Editing using Generative Models

In [ ]:
# Stable Diffusion XL local generation (GPU)
import torch
from diffusers import StableDiffusionXLPipeline
pipe = StableDiffusionXLPipeline.from_pretrained('stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16, use_safetensors=True).to('cuda')
sd_prompt = 'A realistic portrait of a woman in a field of roses'
sd_image = pipe(sd_prompt, num_inference_steps=50, guidance_scale=7.5).images[0]
sd_image.show()

In [ ]:
# Style transfer – painterly version
import matplotlib.pyplot as plt
paint_prompt = sd_prompt + ' in a painterly style'
painted_image = pipe(paint_prompt, num_inference_steps=30, guidance_scale=5.0).images[0]
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(sd_image)
plt.title('Original')
plt.axis('off')
plt.subplot(1,2,2)
plt.imshow(painted_image)
plt.title('Painterly')
plt.axis('off')
plt.show()

In [ ]:
import torch
from diffusers import ZImagePipeline

# 1. Load the pipeline
# Use bfloat16 for optimal performance on supported GPUs
pipe = ZImagePipeline.from_pretrained(
    "Tongyi-MAI/Z-Image-Turbo",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=False,
)
pipe.to("cuda")

# [Optional] Attention Backend
# Diffusers uses SDPA by default. Switch to Flash Attention for better efficiency if supported:
# pipe.transformer.set_attention_backend("flash")    # Enable Flash-Attention-2
# pipe.transformer.set_attention_backend("_flash_3") # Enable Flash-Attention-3

# [Optional] Model Compilation
# Compiling the DiT model accelerates inference, but the first run will take longer to compile.
# pipe.transformer.compile()

# [Optional] CPU Offloading
# Enable CPU offloading for memory-constrained devices.
# pipe.enable_model_cpu_offload()

prompt = "Young Chinese woman in red Hanfu, intricate embroidery. Impeccable makeup, red floral forehead pattern. Elaborate high bun, golden phoenix headdress, red flowers, beads. Holds round folding fan with lady, trees, bird. Neon lightning-bolt lamp (⚡️), bright yellow glow, above extended left palm. Soft-lit outdoor night background, silhouetted tiered pagoda (西安大雁塔), blurred colorful distant lights."

# 2. Generate Image
image = pipe(
    prompt=prompt,
    height=1024,
    width=1024,
    num_inference_steps=9,  # This actually results in 8 DiT forwards
    guidance_scale=0.0,     # Guidance should be 0 for the Turbo models
    generator=torch.Generator("cuda").manual_seed(42),
).images[0]

# image.save("example.png")
image

In [ ]:
plt.imshow(sd_image)

In [ ]:
from diffusers import StableDiffusionXLInpaintPipeline

pipe_inpaint = StableDiffusionXLInpaintPipeline.from_pretrained("diffusers/stable-diffusion-xl-1.0-inpainting-0.1", torch_dtype=torch.float16).to('cuda')

In [ ]:
# Inpainting example – add a small blue ball
from PIL import Image, ImageDraw

# Create mask (ellipse)
mask = Image.new('L', sd_image.size, 0)
# mask = Image.new("L", sd_image.size, 255)
draw = ImageDraw.Draw(mask)
# draw.ellipse([sd_image.width//4, sd_image.height//3, sd_image.width*3//4, sd_image.height*2//3], fill=255)


radius = 200
cx = sd_image.width // 2
cy = sd_image.height // 2

draw.ellipse(
    (cx - radius, cy - radius,
     cx + radius, cy + radius),
    fill=255
)


inpaint_prompt = 'Add smile'

result = pipe_inpaint(
    prompt=inpaint_prompt,
    image=sd_image,
    mask_image=mask,
)

result_image = result.images[0]



plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.imshow(sd_image)
plt.title('Original')
plt.axis('off')
plt.subplot(1,3,2)
plt.imshow(mask)
plt.title('Mask')
plt.axis('off')
plt.subplot(1,3,3)
plt.imshow(result_image)
plt.title('Inpainted')
plt.axis('off')
plt.show()

In [ ]:
from diffusers import StableDiffusionXLImg2ImgPipeline
import torch

pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",
    torch_dtype=torch.float16
).to("cuda")



In [ ]:
# "Turn this daytime street into a snowy winter scene"
# "A watercolor painting of the same scene"

result = pipe(
    prompt="Turn this daytime street into a snowy winter scene",
    image=sd_image,
    strength=0.5
)

result_image = result.images[0]

## Model Unloading and Object Cleaning

In [ ]:
import gc
import torch

# Delete model reference
# del pipe
del pipe_inpaint
# Collect Python garbage
gc.collect()

# Clear PyTorch CUDA cache
torch.cuda.empty_cache()